## Goal

The goal of this project to rank borrowers listed from the Kaggle's "Give Me Some Credit" by risk of default and to detect potential defaults. To be more specific, this project aims to detect serious credit delinquency. In other words, becoming 90 days past due or worse on payments. Predicting the probability of delinquency can assist financial institutions in evaluating credit risk and making lending decisions. 

### Methodology

Since the target variable is binary and the dataset is labeled, our group tested the effectiveness of several models and ranked them according to the ROC AUC metric, which is well-suited for imbalanced binary classification. To be more specific, our group tested the following models:

- Naive Baseline
- SVM (Hard & Soft Margin)
- Logistic Regression
- Gradient Boosting
- Decision Tree
- Random Forest
- MLP

After, we took the best model, ranked by ROC AUC and conducted threshold tuning in order to detect potential defaults, and investigated the overall performance by analyzing the model's metrics like recall, F1 score, and the confusion matrix. 



## Data Cleaning Process

To predict the probability that a borrower will experience financial distress within the next two years, we start by profiling each feature, visually and mathematically. If features are skewed, we use the skewness to route that feature to a different pipeline that can handle the skew accordingly by, for example, selecting the correct power transformer per feature. 

In [ ]:
#| echo: false
from abc import ABC, abstractmethod
from dataclasses import dataclass
from enum import Enum, auto
from functools import cached_property
from itertools import combinations
import random
from typing import ClassVar, Protocol

from great_tables import GT
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import loguniform, norm, normaltest, randint, uniform
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, ConfusionMatrixDisplay, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, PowerTransformer, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier


SEED: int = 42
TARGET: str = "SeriousDlqin2yrs"

random.seed(SEED)
np.random.seed(SEED)

df_train = pd.read_csv("data/cs-training.csv", index_col=0)
df_test  = pd.read_csv("data/cs-test.csv",  index_col=0)

features: list[str] = [c for c in df_train.columns if c != TARGET]

## Missing Value Profiling

Algorithms like SVM and MLP will throw fatal errors on a single `NaN`, so the first step is to find missing values and determine the correct treatment. If the data is missing completely at random (MCAR), we use median imputation since the "missingness" is unrelated to the underlying values. When data is missing not at random (MNAR), for instance, an unemployed borrower omitting `MonthlyIncome`, the missingness is caused by the value itself, and median imputation introduces bias. In that case, we investigate why the value is missing before imputing a value. Regardless, the imputation step must be chained before we perform any transformation to prevent errors. 

In [ ]:
#| echo: false
def profile_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Produce a column-level summary: dtype, null count, null %, and basic stats.

    Args:
        df: The DataFrame to profile.

    Returns:
        Summary DataFrame with one row per column.
    """
    null_count = df.isnull().sum()
    null_pct   = (null_count / len(df) * 100).round(2)
    return pd.DataFrame({
        "dtype"     : df.dtypes,
        "null_count": null_count,
        "null_%"    : null_pct,
        "mean"      : df.mean(numeric_only=True).round(4),
        "std"       : df.std(numeric_only=True).round(4),
        "min"       : df.min(numeric_only=True),
        "max"       : df.max(numeric_only=True),
    })


GT(profile_dataframe(df_train).reset_index().rename(columns={"index": "feature"}))

In [ ]:
#| echo: false
def test_mcar(df: pd.DataFrame, cols: list[str], target: str) -> pd.DataFrame:
    """Test whether missingness in each column is associated with the target variable.

    A statistically significant point-biserial correlation between a missing
    indicator and the target is evidence against MCAR, suggesting MAR or MNAR.

    Args:
        df: Training DataFrame.
        cols: Columns to test for informative missingness.
        target: Binary target column name.

    Returns:
        DataFrame with default rates, correlation, and p-value per column.
    """
    from scipy.stats import pointbiserialr

    rows = []
    for col in cols:
        indicator = df[col].isna().astype(int)
        r, p = pointbiserialr(indicator, df[target])
        rows.append({
            "column": col,
            "missing_pct": round(indicator.mean() * 100, 2),
            "default_rate_observed": round(df.loc[indicator == 0, target].mean(), 4),
            "default_rate_missing": round(df.loc[indicator == 1, target].mean(), 4),
            "r": round(r, 4),
            "p_value": round(p, 10),
        })
    return pd.DataFrame(rows)

missing_cols = [c for c in features if df_train[c].isna().any()]
mcar_df = test_mcar(df_train, missing_cols, TARGET)
(
    GT(mcar_df)
    .tab_header(
        title="MCAR Test — Point-Biserial Correlation",
        subtitle="Is missingness associated with the target variable?"
    )
    .fmt_number(columns=["missing_pct", "default_rate_observed", "default_rate_missing", "r"], decimals=4)
    .fmt_scientific(columns="p_value")
)

Since both columns return p-values below 0.05, we reject MCAR. 

What makes this a genuine surprise is the direction: rows with missing values show a lower default rate, not higher. The intuitive assumption would be the opposite — that missing income signals financial instability and higher risk. Instead, the data suggests the missing group may skew toward retired borrowers with no employment income to report, but also lower credit risk. As a result we consider this a meaningful signal, even though it's unexpected. The correct treatment is to first create a binary flag for each affected column — so we can preserve the meaningful signal and also impute the median to satisfy algorithm requirements.

We now apply the two-step fix derived above, creating the binary flags before imputation so no signal is overwritten by the fill.

In [ ]:
#| echo: false
MONTHLY_INCOME_MEDIAN: float = df_train["MonthlyIncome"].median()
DEPENDENTS_MEDIAN: float = df_train["NumberOfDependents"].median()


def add_missing_flags(df: pd.DataFrame) -> pd.DataFrame:
    """Add boolean missing-value indicator columns before imputation.

    Flags are derived before filling so the signal is not overwritten.
    Uses integer encoding (0/1) for direct compatibility with sklearn estimators.

    Args:
        df: DataFrame to annotate.

    Returns:
        DataFrame with IsMonthlyIncomeMissing and IsNumberOfDependentsMissing appended.
    """
    return df.assign(
        IsMonthlyIncomeMissing=df["MonthlyIncome"].isna().astype(int),
        IsNumberOfDependentsMissing=df["NumberOfDependents"].isna().astype(int),
    )


def impute_missing(df: pd.DataFrame) -> pd.DataFrame:
    """Impute missing values using training-set medians.

    MonthlyIncome is filled with the training median.
    NumberOfDependents is filled with 0, which equals the training median
    (59% of observed values are 0) and aligns with the retiree hypothesis.

    Args:
        df: DataFrame with missing-value flag columns already added.

    Returns:
        DataFrame with NaNs in MonthlyIncome and NumberOfDependents resolved.
    """
    return df.fillna({
        "MonthlyIncome": MONTHLY_INCOME_MEDIAN,
        "NumberOfDependents": DEPENDENTS_MEDIAN,
    })


df_train = impute_missing(add_missing_flags(df_train))
features = [c for c in df_train.columns if c != TARGET]
df_train[["MonthlyIncome", "IsMonthlyIncomeMissing",
          "NumberOfDependents", "IsNumberOfDependentsMissing"]].describe().T

## Class Imbalance — Target Variable

Now we examine the distribution of `SeriousDlqin2yrs`. Since large disparities between classes allow naive models to achieve high accuracy by just predicting the majority class, we want to proactively check for the imbalance in the training set only. If severe imbalance is detected, two remedies are available:

1. Synthesize minority-class examples (SMOTE): Create datapoints in the training fold to balance the class ratio.
2. Update class weights: by updating the weights of the imbalanced class we can increase the consequence of misclassifying the minority class.

In [ ]:
#| echo: false
def check_class_balance(df: pd.DataFrame, target: str) -> pd.DataFrame:
    """Compute class distribution for the target variable.

    Args:
        df: Training DataFrame.
        target: Name of the target column.

    Returns:
        DataFrame with class label, count, and percentage.
    """
    counts = df[target].value_counts().rename_axis("class").reset_index(name="count")
    counts["pct"] = (counts["count"] / counts["count"].sum() * 100).round(2)
    return counts

class_balance_df = check_class_balance(df_train, TARGET)
(
    GT(class_balance_df)
    .tab_header(title="Class Distribution", subtitle=TARGET)
    .fmt_integer(columns="count")
    .fmt_number(columns="pct", decimals=2)
)

## Distribution Analysis

### Visual Check

Below are the histograms for every continuous feature to give us a look at each feature's shape before formal testing. We are looking for heavy right tails, narrow spikes at zero, or multi-modal shapes — any of which indicate deviation from normality that could destabilise distance-based and gradient-descent models.

In [ ]:
#| echo: false
df_train[features].hist(bins=60, figsize=(15, 10), layout=(4, 3))
plt.suptitle("Feature Distributions — Training Set", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

### Mathematical Proof (D'Agostino K²)

To formally test whether each feature follows a normal distribution, we apply D'Agostino's $K^2$ test, which uses skewness and kurtosis into a single statistic which we can use to disprove the null hypothesis, which is: the feature originates from a normal distribution. A p-value below 0.05 gives grounds to reject $H_0$, confirming the feature is non-normal. If we fail to reject, we do not require power transformation. If non-normality is widespread, standard scaling alone will be insufficient and the pipeline must rely on power transformations to address the underlying distribution shape.

In [ ]:
#| echo: false
def run_normality_tests(df: pd.DataFrame, features: list[str], alpha: float = 0.05) -> pd.DataFrame:
    """Run D'Agostino's K² normality test on each feature and report results.

    Args:
        df:       DataFrame containing the features.
        features: Column names to test.
        alpha:    Significance level for rejecting H0 (default 0.05).

    Returns:
        DataFrame with columns: feature, statistic, p_value, reject_H0, verdict.
    """
    records = [
        {
            "feature"  : col,
            "statistic": round(stat, 4),
            "p_value"  : round(p, 6),
            "reject_H0": p < alpha,
            "verdict"  : "Non-normal" if p < alpha else "Normal",
        }
        for col in features
        for stat, p in [normaltest(df[col].dropna())]
    ]
    return pd.DataFrame(records)


GT(run_normality_tests(df_train, features))

### Data Quality — DebtRatio Encoding Inconsistency

`DebtRatio` is defined as monthly debt payments divided by monthly income. If `MonthlyIncome` is missing, a valid ratio should be undefined. However, it appears that every row with missing income somehow carries a non-null `DebtRatio`. Below, we examine how the column is encoding something different for those rows by comparing its distribution between the two groups. If there is a massive divergence, it likely means the column is storing two completely different types of data.

In [ ]:
#| echo: false
def compare_groups_by_missingness(
    df: pd.DataFrame, flag_col: str, compare_features: list[str]
) -> pd.DataFrame:
    """Compare feature medians between rows where flag_col is 1 vs 0.

    Args:
        df: Training DataFrame.
        flag_col: Binary indicator column (1 = was missing, 0 = was present).
        compare_features: Features to summarise across the two groups.

    Returns:
        DataFrame with per-feature medians for each group and their ratio.
    """
    is_missing = df[flag_col].astype(bool)
    rows = []
    for col in compare_features:
        med_present = df.loc[~is_missing, col].median()
        med_missing = df.loc[is_missing, col].median()
        ratio = (
            round(med_missing / med_present, 2)
            if med_present not in (0, None) and pd.notna(med_present) and pd.notna(med_missing)
            else None
        )
        rows.append({
            "feature": col,
            "median_income_present": med_present,
            "median_income_missing": med_missing,
            "ratio": ratio,
        })
    return (
        pd.DataFrame(rows)
        .sort_values("ratio", ascending=False, na_position="last")
        .reset_index(drop=True)
    )

non_target_features = [c for c in df_train.columns if c != TARGET]
group_df = compare_groups_by_missingness(df_train, "IsMonthlyIncomeMissing", non_target_features)
(
    GT(group_df)
    .tab_header(
        title="Feature Medians — MonthlyIncome Missing vs. Present",
        subtitle="Sorted by ratio (missing / present) descending — ratio column omitted where present median is 0"
    )
    .fmt_number(columns=["median_income_present", "median_income_missing", "ratio"], decimals=4)
)

In [ ]:
#| echo: false
def plot_debtratio_before_after(df: pd.DataFrame, flag_col: str, col: str = "DebtRatio") -> None:
    """Plot DebtRatio distribution before and after isolating the corrupted rows.

    Args:
        df: Training DataFrame with binary missing-income flag.
        flag_col: Binary indicator for missing income (1 = missing).
        col: Feature column to plot.
    """
    is_corrupted = df[flag_col].astype(bool)
    before = df[col]
    after  = df.loc[~is_corrupted, col]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(
        "DebtRatio — Before and After Removing Corrupted Rows",
        fontsize=12, fontweight="bold"
    )

    axes[0].hist(before.clip(upper=before.quantile(0.995)), bins=80, color="steelblue", alpha=0.8)
    axes[0].set_title("All rows (clipped at 99.5th percentile for visibility)")
    axes[0].set_xlabel("DebtRatio")
    axes[0].set_ylabel("count")

    axes[1].hist(after.clip(upper=5), bins=80, color="darkorange", alpha=0.8)
    axes[1].set_title("Income-present rows only (clipped at 5 for visibility)")
    axes[1].set_xlabel("DebtRatio")

    plt.tight_layout()
    plt.show()


plot_debtratio_before_after(df_train, "IsMonthlyIncomeMissing")

`DebtRatio` is the only feature that diverges — its median is more than 3,900 times higher in the missing-income group (1,159 vs 0.296). 

While a debt-to-income ratio above 1.0 is technically possible, values in the hundreds or thousands should be flagged. The 75th percentile sits at 0.87, then the 90th jumps to 1,267 — that is not a heavy tail, it is two different encodings in the same column. For the missing-income group, the column is likely storing raw monthly debt obligations in dollars rather than a computed ratio.

This reinforces the retired-borrower hypothesis: the missing-income group is older (median age 57 vs 51) and carries lower revolving utilisation (0.08 vs 0.18). Since `IsMonthlyIncomeMissing` already captures the structural difference of these rows, the cleanest action is to drop `DebtRatio` entirely — removing a corrupted feature without discarding any information not already represented more reliably elsewhere.

### Data Quality — RevolvingUtilization Extreme Values

`RevolvingUtilizationOfUnsecuredLines` should be a ratio between 0 and 1, representing credit used divided by credit limit. Values marginally above 1.0 are plausible, since fees and interest can push balances past the stated limit, but we inspect the upper tail for the same dollar-encoding issue found in `DebtRatio`. A cliff in the percentile distribution — where the column transitions from ratio-scale values to implausibly large numbers — would confirm the problem.

In [ ]:
#| echo: false
def profile_upper_tail(
    df: pd.DataFrame, col: str, percentiles: list[float] | None = None
) -> pd.DataFrame:
    """Summarise the upper tail of a column to surface encoding discontinuities.

    Args:
        df: Training DataFrame.
        col: Column to profile.
        percentiles: Percentile breakpoints to include. Defaults to standard set.

    Returns:
        DataFrame of percentile labels and their corresponding values.
    """
    pcts = percentiles or [0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1.0]
    labels = [f"p{int(p * 100)}" if p < 1 else "max" for p in pcts]
    values = df[col].quantile(pcts).values
    return pd.DataFrame({"percentile": labels, "value": values})

tail_df = profile_upper_tail(df_train, "RevolvingUtilizationOfUnsecuredLines")
(
    GT(tail_df)
    .tab_header(
        title="RevolvingUtilizationOfUnsecuredLines — Upper Tail",
        subtitle="Identifying the encoding cliff"
    )
    .fmt_number(columns="value", decimals=4)
)

The cliff is unmistakable. The 90th percentile is 0.98, the 95th is 1.0, and the 99.9th jumps to 1,571 with a maximum of 50,708. While values slightly above 1.0 are defensible, values in the thousands must be dollar balances, not ratios — the same encoding inconsistency found in `DebtRatio`. Unlike `DebtRatio`, however, the feature is well-behaved for the vast majority of rows, so dropping it would discard real signal. Capping at the 99th percentile (~1.09) preserves legitimate near-limit values while suppressing the dollar-encoded outliers.

### Data Quality — Delinquency Column Error Codes

The three delinquency count columns showed a suspicious gap in their value distributions — counts jump from plausible values (0–17) directly to 96 and 98, with nothing in between. In financial survey datasets, values such as 96, 98, and 99 are commonly used as error codes to flag records as not applicable or erroneous rather than genuine counts. We investigate whether the same rows carry these anomalous values across all three columns simultaneously, which would confirm they are error codes rather than real delinquency counts.

In [ ]:
#| echo: false
def find_error_code_rows(
    df: pd.DataFrame, cols: list[str], error_codes: list[int]
) -> pd.DataFrame:
    """Identify rows carrying error codes and verify cross-column consistency.

    Args:
        df: Training DataFrame.
        cols: Columns expected to share the error code encoding.
        error_codes: Candidate error codes to check.

    Returns:
        DataFrame summarising error code counts and cross-column consistency.
    """
    rows = []
    for val in error_codes:
        masks = [df[c] == val for c in cols]
        count = masks[0].sum()
        all_identical = all((masks[0] == m).all() for m in masks[1:])
        rows.append({
            "error_code": val,
            "row_count": int(count),
            "identical_across_all_columns": all_identical,
        })
    return pd.DataFrame(rows)

delinquency_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]
error_code_df = find_error_code_rows(df_train, delinquency_cols, error_codes=[96, 98])
(
    GT(error_code_df)
    .tab_header(
        title="Delinquency Column Error Codes",
        subtitle="Do the same rows carry the same anomalous value across all three columns?"
    )
    .fmt_integer(columns=["error_code", "row_count"])
)

In the analysis, we see that 96 and 98 are displayed in exactly the same rows across all three delinquency columns. Given that 269 real borrowers with exactly identical counts deliquencies from 30–59, 60–89, and 90+ days is highly unlikely, we'll assume that the values 96 and 98 are indeed error codes. The cleanest treatment is to replace these values with `NaN` and drop the affected rows outright — imputing delinquency counts for corrupted records would risk introducing more noise than signal.

### Data Quality — MonthlyIncome Extreme Values

`MonthlyIncome` has a maximum of \$3,008,750 per month, roughly 38 times the 99.9th percentile. Before accepting these as legitimate, we check whether extreme income values are associated with lower default rates and whether they are concentrated enough to distort downstream transformations. We are looking both for a clear relationship between income and default risk and for a distributional cliff similar to those found in `DebtRatio` and `RevolvingUtilization`.

In [ ]:
#| echo: false
def default_rate_by_income_bucket(
    df: pd.DataFrame, income_col: str, target: str
) -> pd.DataFrame:
    """Compute default rate, row count, and median income across income brackets.

    Args:
        df: Training DataFrame.
        income_col: Name of the income column.
        target: Binary target column name.

    Returns:
        DataFrame with one row per income bracket.
    """
    df_valid = df.dropna(subset=[income_col]).copy()
    df_valid["income_bucket"] = pd.cut(
        df_valid[income_col],
        bins=[0, 2_000, 5_000, 10_000, 20_000, 50_000, float("inf")],
        labels=["<$2k", "$2k–$5k", "$5k–$10k", "$10k–$20k", "$20k–$50k", ">$50k"],
    )
    return (
        df_valid.groupby("income_bucket", observed=True)
        .agg(
            row_count=(income_col, "count"),
            median_income=(income_col, "median"),
            default_rate=(target, "mean"),
        )
        .round(4)
        .reset_index()
    )

income_bucket_df = default_rate_by_income_bucket(df_train, "MonthlyIncome", TARGET)
(
    GT(income_bucket_df)
    .tab_header(
        title="Default Rate by Monthly Income Bracket",
        subtitle="Rows with missing MonthlyIncome excluded"
    )
    .fmt_integer(columns="row_count")
    .fmt_currency(columns="median_income", currency="USD")
    .fmt_percent(columns="default_rate", decimals=2)
)

The default rate falls sharply from 9.1% at under \$2k per month to 4.2% at \$10k–\$20k, then slightly increases for the highest earners (5.3% at \$20k–\$50k, 5.7% above \$50k). In other words, very high earners are more likely carry leverage or take on riskier debt structures. So, the assumption that wealth implies a more reliable borrower doesn't hold at the extreme end.

The conclusion: the extreme values are real, not encoding errors — they are demographically coherent and all non-defaulting. However, a handful of values which are 38 times the 99.9th percentile will dominate the Yeo-Johnson lambda fit for the Branch B models. For the tree models in Branch A, we won't cap these values since trees split on thresholds and handle extreme values natively. For Branch B, `MonthlyIncome` is capped at the 99.9th percentile (~\$78k/month) before transformation, applied only after the `IsMonthlyIncomeMissing` flag and imputation steps are complete.

## Multicollinearity — Correlation Check

After resolving the quality issues of the individual columns, we also need to ensure our features aren't just echoing the same information. To do this, we compute the Pearson correlation matrix for all features, rank them, and inspect the most correlated pairs (in this case 10). A highly collinear pair is measuring the same underlying signal, so retaining both adds noise without adding information. Pairs with $|r| > 0.8$ are candidates for removal. For SVM and MLP, redundant features increase the runtime complexity and also increase overfitting risk. For tree models, feature importance scores become artificially diluted across collinear features, rendering them uninterpretable. The appropriate response is to drop the redundant feature from each flagged pair, retaining the more informative signal.

In [ ]:
#| echo: false
def top_correlations(
    df: pd.DataFrame, features: list[str], n: int = 10
) -> pd.DataFrame:
    """Return the top N most correlated feature pairs by absolute Pearson r.

    Args:
        df: Training DataFrame.
        features: Feature columns to evaluate.
        n: Number of top pairs to return.

    Returns:
        DataFrame of feature pairs sorted by absolute correlation, descending.
    """
    corr = df[features].corr(method="pearson")
    pairs = (
        corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
    )
    pairs.columns = ["feature_a", "feature_b", "correlation"]
    return (
        pairs.reindex(pairs["correlation"].abs().sort_values(ascending=False).index)
        .head(n)
        .reset_index(drop=True)
    )

top_corr_df = top_correlations(df_train, features)
(
    GT(top_corr_df)
    .tab_header(title="Top 10 Feature Correlations", subtitle="Pearson r — sorted by absolute value")
    .fmt_number(columns="correlation", decimals=4)
)

## Skewness Heuristic — Pipeline Branching

Now that we've identified the critical properties that could cause our models to fail, the next step is to programmatically route features for pre-processing. Since our data shows class imbalance, we compute the absolute skewness score for each feature and apply a rule: if a feature has $|\text{skew}| > 1$ we route the feature to Branch B for transformation and scaling (normalization). Features with low skew scores can safely bypass power transformation; those above the threshold require treatment before distance-based or gradient-descent models can use them effectively.

In [ ]:
#| echo: false
def compute_skewness_table(df: pd.DataFrame, features: list[str], threshold: float = 1.0) -> pd.DataFrame:
    """Compute skewness for each feature and assign a pipeline branch.

    Args:
        df:        DataFrame containing the features.
        features:  Column names to evaluate.
        threshold: Absolute skew above which a feature is flagged (default 1.0).

    Returns:
        DataFrame sorted by abs_skew descending.
    """
    records = [
        {
            "feature"      : col,
            "skewness"     : round(df[col].skew(), 4),
            "abs_skew"     : round(abs(df[col].skew()), 4),
            "highly_skewed": abs(df[col].skew()) > threshold,
            "branch"       : "B — Transform + Scale" if abs(df[col].skew()) > threshold else "A — Raw (tree-safe)",
        }
        for col in features
    ]
    return pd.DataFrame(records).sort_values("abs_skew", ascending=False)


skewness_df = compute_skewness_table(df_train, features)
GT(skewness_df)

## Transformer Selection — Box-Cox vs. Yeo-Johnson

Finally once we've determined which features need to be routed, we need to decide which how they'll be transformed. In essence, we'd like to map our skewed distribution into something that's closer to a normal distribution. For this we use what's called a Power transformer and choose between the following options:
1. Box-Cox: works only with strictly positive values.
2. Yeo-Johnson: works on any value including zero and negatives.

So before applying any transformation we inspect the minimum value of each Branch B feature to select the correct algorithm. The decision rule is: if $\min(\text{feature}) > 0$, use Box-Cox; if $\min(\text{feature}) \le 0$, use Yeo-Johnson.

While Yeo-Johnson safely handles negatives and zeros, defaulting to it unconditionally incurs a performance penalty. If the data allows, we prefer to use a Box-Cox transformation to reduce the pipeline's overhead.

In [ ]:
#| echo: false
def select_transformer(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """Assign a power transformer to each feature based on its minimum value.

    Decision rule:
      min > 0  + Box-Cox (strictly positive data)
      min <= 0 + Yeo-Johnson (handles zeros and negatives)

    Args:
        df:       DataFrame containing the features.
        features: Column names to evaluate (typically the highly-skewed subset).

    Returns:
        DataFrame with columns: feature, min_value, transformer, reason.
    """
    records = [
        {
            "feature"    : col,
            "min_value"  : df[col].min(),
            "transformer": "Box-Cox" if df[col].min() > 0 else "Yeo-Johnson",
            "reason"     : "min > 0 — strictly positive" if df[col].min() > 0
                           else "min ≤ 0 — contains zeros or negatives",
        }
        for col in features
    ]
    return pd.DataFrame(records)


highly_skewed_features = skewness_df.loc[skewness_df["highly_skewed"], "feature"].tolist()
transformer_plan = select_transformer(df_train, highly_skewed_features)
GT(transformer_plan)

### Pipeline

Our diagnosis established four core data quality treatments followed by a clear branching strategy. We encoded these steps into a reproducible sklearn pipeline fitted strictly on the training set. The execution order is as follows:

1. **Cleaning Error Codes:** Replace the 96 and 98 error codes with `NaN` and drop the affected rows for both branches.
2. **Removing Corrupted Data:** Drop `DebtRatio` entirely since the encoding was corrputed.
3. **Reducing Collinearity:** Drop the two delinquency columns that exhibit collinearity with `NumberOfTimes90DaysLate`.
4. **Capping Extreme Values:** Cap `RevolvingUtilizationOfUnsecuredLines` at the 99.9th percentile.
5. **Routing** 
    -  **Branch A (tree models):** Pass the newly cleaned features through completely raw and unscaled.
    -  **Branch B:** Cap `MonthlyIncome` at the 99.9th percentile, apply a Yeo-Johnson transformation, and then pass all skewed features through a `StandardScaler`.

In [ ]:
#| echo: true

class FeatureTransformer(Protocol):
    """Interface for per-feature Branch B transformations."""

    def __str__(self) -> str: ...
    def pipeline(self) -> BaseEstimator: ...


class BinaryPassthrough(FeatureTransformer):
    """Identity transform for binary indicator features — preserves 0/1 encoding unchanged."""

    def __str__(self) -> str:
        return "passthrough"

    def pipeline(self) -> FunctionTransformer:
        return FunctionTransformer()


class SymmetricTransformer(FeatureTransformer):
    """Standard scaling only, for low-skew continuous features that need centering but not reshaping."""

    def __str__(self) -> str:
        return "StandardScaler"

    def pipeline(self) -> StandardScaler:
        return StandardScaler()


class SkewedTransformer(FeatureTransformer):
    """Yeo-Johnson followed by standard scaling, for highly skewed features without extreme outliers."""

    def __str__(self) -> str:
        return "Yeo-Johnson + StandardScaler"

    def pipeline(self) -> Pipeline:
        return Pipeline([
            ("power", PowerTransformer(method="yeo-johnson")),
            ("scaler", StandardScaler()),
        ])


class CappedSkewedTransformer(SkewedTransformer):
    """Extends SkewedTransformer by prepending a percentile cap for features with extreme outliers.

    Args:
        cap: Upper bound to clip values to before the Yeo-Johnson transform.
    """

    def __init__(self, *, cap: float) -> None:
        self.cap = cap

    def __str__(self) -> str:
        return "cap \u2192 Yeo-Johnson + StandardScaler"

    def pipeline(self) -> Pipeline:
        return Pipeline([
            ("cap", FunctionTransformer(lambda X: np.clip(X, None, self.cap))),
            *super().pipeline().steps,
        ])




REVOLVING_CAP: float = df_train["RevolvingUtilizationOfUnsecuredLines"].quantile(0.999)
INCOME_CAP: float = df_train["MonthlyIncome"].quantile(0.999)
FEATURE_PIPELINE_MAP: dict[str, FeatureTransformer] = {
    "RevolvingUtilizationOfUnsecuredLines": CappedSkewedTransformer(cap=REVOLVING_CAP),
    "age":                                  SymmetricTransformer(),
    "MonthlyIncome":                        CappedSkewedTransformer(cap=INCOME_CAP),
    "NumberOfOpenCreditLinesAndLoans":      SkewedTransformer(),
    "NumberOfTimes90DaysLate":              SkewedTransformer(),
    "NumberRealEstateLoansOrLines":         SkewedTransformer(),
    "NumberOfDependents":                   SkewedTransformer(),
    "IsMonthlyIncomeMissing":               BinaryPassthrough(),
    "IsNumberOfDependentsMissing":          BinaryPassthrough(),
}

df_clean = (
    df_train
    .assign(**{"NumberOfTime30-59DaysPastDueNotWorse": lambda x: x["NumberOfTime30-59DaysPastDueNotWorse"].replace(96, np.nan)})
    .assign(**{"NumberOfTime30-59DaysPastDueNotWorse": lambda x: x["NumberOfTime30-59DaysPastDueNotWorse"].replace(98, np.nan)})
    .assign(**{"NumberOfTime60-89DaysPastDueNotWorse": lambda x: x["NumberOfTime60-89DaysPastDueNotWorse"].replace(96, np.nan)})
    .assign(**{"NumberOfTime60-89DaysPastDueNotWorse": lambda x: x["NumberOfTime60-89DaysPastDueNotWorse"].replace(98, np.nan)})
    .assign(NumberOfTimes90DaysLate=lambda x: x["NumberOfTimes90DaysLate"].replace(96, np.nan))
    .assign(NumberOfTimes90DaysLate=lambda x: x["NumberOfTimes90DaysLate"].replace(98, np.nan))
    .dropna(subset=["NumberOfTimes90DaysLate"])
    .reset_index(drop=True)
)

In [ ]:
#| echo: false
(
    GT(pd.DataFrame([
        {"feature": col, "branch_a": "raw", "branch_b": str(tx)}
        for col, tx in FEATURE_PIPELINE_MAP.items()
    ]))
    .tab_header(title="Feature Routing", subtitle="Treatment applied per branch")
)

## Train / Validation Split

The test set labels are not publicly available — evaluating against them requires uploading predictions through the Kaggle API, which is outside the scope of this local analysis. To measure model performance without submitting, we reserve 20% of the training data as a held-out validation set.

Because the target is heavily imbalanced (roughly 6.7% default rate), the split is stratified: passing `stratify=_y` to `train_test_split` ensures both halves retain the original class ratio. Without stratification, random sampling could accidentally concentrate defaults into one partition and produce misleading performance estimates.

In [ ]:
VAL_FRAC: float        = 0.20
_X:       pd.DataFrame = df_clean[list(FEATURE_PIPELINE_MAP.keys())]
_y:       pd.Series    = df_clean[TARGET]


@dataclass(eq=False)
class Split:
    """Feature matrix and target vector for one dataset partition."""
    X: pd.DataFrame
    y: pd.Series


class Dataset:
    """Training and validation splits for one preprocessing branch."""

    def __init__(
        self,
        raw_X: pd.DataFrame,
        raw_y: pd.Series,
        *,
        test_size: float = 0.2,
        random_state: int | None = None,
        stratify: pd.Series | None = None,
    ) -> None:
        """Split raw_X and raw_y and construct a Dataset.

        Args:
            raw_X: Full feature DataFrame before splitting.
            raw_y: Full target Series before splitting.
            test_size: Fraction of rows reserved for the validation split.
            random_state: Random seed forwarded to train_test_split.
            stratify: Series passed to train_test_split to preserve class balance.
        """
        X_tr, X_val, y_tr, y_val = train_test_split(raw_X, raw_y, test_size=test_size, random_state=random_state, stratify=stratify)
        self.train = Split(X=X_tr.reset_index(drop=True),  y=y_tr.reset_index(drop=True))
        self.val   = Split(X=X_val.reset_index(drop=True), y=y_val.reset_index(drop=True))

    @classmethod
    def pipelined(
        cls,
        raw_X: pd.DataFrame,
        raw_y: pd.Series,
        *,
        pipeline: TransformerMixin,
        test_size: float = 0.2,
        random_state: int | None = None,
        stratify: pd.Series | None = None,
    ) -> "Dataset":
        """Split raw_X and raw_y, apply pipeline, then delegate to __init__.

        Args:
            raw_X: Full feature DataFrame before splitting.
            raw_y: Full target Series before splitting.
            pipeline: sklearn TransformerMixin fitted on the training half
                and applied to both halves — accepts any TransformerMixin
                (ColumnTransformer, Pipeline, etc.).
            test_size: Fraction of rows reserved for the validation split.
            random_state: Random seed forwarded to train_test_split.
            stratify: Series passed to train_test_split to preserve class balance.

        Returns:
            Dataset with .train and .val Split objects populated.
        """
        dataset = cls(raw_X, raw_y, test_size=test_size, random_state=random_state, stratify=stratify)
        cols = list(raw_X.columns)
        X_tr_pipelined  = pd.DataFrame(pipeline.fit_transform(dataset.train.X), columns=cols)
        X_val_pipelined = pd.DataFrame(pipeline.transform(dataset.val.X),       columns=cols)
        dataset.train.X = X_tr_pipelined
        dataset.val.X   = X_val_pipelined
        return dataset


class Data(Enum):
    """Dataset variants keyed by preprocessing branch.

    Each member's value is a Dataset with .train and .val Split objects.
    Split.X is the feature matrix; Split.y is the target vector.
    All preprocessors are fitted on .train only; .val is never seen during fitting.
    """
    CLEAN  = Dataset(_X, _y, test_size=VAL_FRAC, random_state=SEED, stratify=_y)
    SCALED = Dataset.pipelined(
        _X, _y,
        pipeline=ColumnTransformer(
            transformers=[(col, tx.pipeline(), [col]) for col, tx in FEATURE_PIPELINE_MAP.items()],
            remainder="drop",
        ),
        test_size=VAL_FRAC, random_state=SEED, stratify=_y,
    )

    @property
    def train(self) -> Split:
        return self.value.train

    @property
    def val(self) -> Split:
        return self.value.val

In [ ]:
#| echo: false
(
    GT(pd.DataFrame([
        {"split": "train", "branch": "A — Tree models", "rows": Data.CLEAN.train.X.shape[0],  "features": Data.CLEAN.train.X.shape[1],  "transformations": "none (raw)"},
        {"split": "val",   "branch": "A — Tree models", "rows": Data.CLEAN.val.X.shape[0],    "features": Data.CLEAN.val.X.shape[1],    "transformations": "none (raw)"},
        {"split": "train", "branch": "B — SVM / MLP",   "rows": Data.SCALED.train.X.shape[0], "features": Data.SCALED.train.X.shape[1], "transformations": "per feature routing"},
        {"split": "val",   "branch": "B — SVM / MLP",   "rows": Data.SCALED.val.X.shape[0],   "features": Data.SCALED.val.X.shape[1],   "transformations": "per feature routing"},
    ]))
    .tab_header(title="Pipeline Output", subtitle="Dataset dimensions per branch and split")
    .fmt_integer(columns=["rows", "features"])
)

In [ ]:
#| echo: false
(
    GT(pd.DataFrame([
        {"branch": "A — Tree models", "rows": Data.CLEAN.train.X.shape[0],  "features": Data.CLEAN.train.X.shape[1],  "transformations": "none (raw)"},
        {"branch": "B — SVM / MLP",   "rows": Data.SCALED.train.X.shape[0], "features": Data.SCALED.train.X.shape[1], "transformations": "per feature routing"},
    ]))
    .tab_header(title="Pipeline Output", subtitle="Training set dimensions per branch")
    .fmt_integer(columns=["rows", "features"])
)

In [ ]:
#| echo: false
def plot_branch_b_distributions(
    df_before: pd.DataFrame,
    df_after: pd.DataFrame,
    features: list[str],
) -> None:
    """Plot Branch B feature distributions before and after pipeline transformation.

    Args:
        df_before: Cleaned but untransformed feature DataFrame.
        df_after: Transformed Branch B DataFrame.
        features: Features to compare — should be the skewed Branch B columns.
    """
    n = len(features)
    fig, axes = plt.subplots(n, 2, figsize=(14, n * 2.5))
    fig.suptitle(
        "Branch B Feature Distributions — Before vs. After Pipeline",
        fontsize=13, fontweight="bold", y=1.01,
    )
    for i, feat in enumerate(features):
        axes[i, 0].hist(df_before[feat].dropna(), bins=60, color="steelblue", alpha=0.8)
        axes[i, 0].set_title(f"{feat}  |  before", fontsize=9)
        axes[i, 0].set_ylabel("count")

        axes[i, 1].hist(df_after[feat], bins=60, color="darkorange", alpha=0.8)
        axes[i, 1].set_title(f"{feat}  |  after Yeo-Johnson + StandardScaler", fontsize=9)

    plt.tight_layout()
    plt.show()


transformed_features = [col for col, tx in FEATURE_PIPELINE_MAP.items() if isinstance(tx, SkewedTransformer)]
plot_branch_b_distributions(Data.CLEAN.train.X, Data.SCALED.train.X, transformed_features)

## Summary

The sections above establish a complete diagnostic picture of the training data. Features with $|\text{skew}| \le 1$ are passed raw and unscaled to the tree models in Branch A. Highly skewed features are imputed, power-transformed, and scaled for the distance and gradient-based models in Branch B. All imputers, transformers, and scalers are fitted only on the training set and applied via `.transform()` to the test set — `.fit()` and `.fit_transform()` are never called on test data.

## Model Evaluation Framework

To succeed at ranking the borrowers, we put eight different models to the test. Although, every model has its own hyperparameters, we tune them on a subsampled training split, pull out the absolute best estimator, and finally score them against the held-out validation set.

| Model | Features | Estimator |
|---|---|---|
| Naive Baseline | Raw | — |
| Logistic Regression | Scaled | `LogisticRegression` |
| SVM — Hard margin | Raw | `SVC` |
| SVM — Hard margin | Scaled | `SVC` |
| SVM — Soft margin | Scaled | `SVC` |
| Gradient Boosting | Raw | `GradientBoostingClassifier` |
| Random Forest | Raw | `RandomForestClassifier` |
| Decision Tree | Raw | `DecisionTreeClassifier` |
| MLP | Scaled | `MLPClassifier` |

To avoid duplicating evaluation scaffolding for every model, we define a `Model` abstract base class. Subclasses declare three things:

- `label` — display name used in tables and plots
- `data` — which preprocessing branch (`Data.CLEAN` or `Data.SCALED`)
- `_build_search()` — returns an unfitted `RandomizedSearchCV` configured for that algorithm, or `None` for the naive baseline

The base class does the rest: subsampling the training split, fitting the search, extracting `best_estimator_`, computing predicted probabilities, and deriving the ROC curve and AUC. All of these are lazy `cached_property` values — nothing runs until first accessed.

For the sake of quick iteration, we've also defined a `FAST_MODE` variable which controls the sample size which we use for tuning. Since the high level discovery and demonstration of knowledge is more important than the exact accuracy of our model, we've left fast mode on to facilitate faster learning. 

In [ ]:
FAST_MODE: bool = True

TUNE_SAMPLE: int = 3_000  if FAST_MODE else 10_000
TUNE_ITER:   int = 5      if FAST_MODE else 20
TUNE_FOLDS:  int = 2      if FAST_MODE else 3


class Model(ABC):
    """Base class for all models in the evaluation pipeline.

    Subclasses declare label, data, and _build_search.
    All evaluation metrics are computed lazily and cached on first access.
    Threshold-dependent metrics use the Youden's J optimal cutoff.
    """

    label: ClassVar[str]
    data:  ClassVar[Data]

    @abstractmethod
    def _build_search(self) -> RandomizedSearchCV | None:
        """Return an unfitted RandomizedSearchCV, or None for the naive baseline.

        Returns:
            Unfitted RandomizedSearchCV, or None to fall back to constant scores.
        """
        ...

    @cached_property
    def search(self) -> RandomizedSearchCV | None:
        """Fit the search on a subsampled training split.

        Returns:
            Fitted RandomizedSearchCV, or None if _build_search returns None.
        """
        s = self._build_search()
        if s is None:
            return None
        idx = np.random.RandomState(SEED).choice(
            len(self.data.train.X), size=min(TUNE_SAMPLE, len(self.data.train.X)), replace=False,
        )
        s.fit(self.data.train.X.iloc[idx], self.data.train.y.iloc[idx])
        return s

    @cached_property
    def estimator(self) -> BaseEstimator:
        """Best estimator from the fitted search."""
        return self.search.best_estimator_

    @cached_property
    def scores(self) -> np.ndarray:
        """Predicted probabilities on the validation split."""
        return self.estimator.predict_proba(self.data.val.X)[:, 1]

    @cached_property
    def _roc(self) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        fpr, tpr, thresholds = roc_curve(self.data.val.y, self.scores)
        return fpr, tpr, thresholds

    @property
    def fpr(self) -> np.ndarray:
        return self._roc[0]

    @property
    def tpr(self) -> np.ndarray:
        return self._roc[1]

    @cached_property
    def auc(self) -> float:
        return roc_auc_score(self.data.val.y, self.scores)

    @cached_property
    def threshold(self) -> float:
        """Optimal classification threshold via Youden's J (argmax TPR - FPR)."""
        fpr, tpr, thresholds = self._roc
        return float(thresholds[np.argmax(tpr - fpr)])

    @cached_property
    def predictions(self) -> np.ndarray:
        """Hard binary predictions at the Youden's J threshold."""
        return (self.scores >= self.threshold).astype(int)

    @cached_property
    def recall(self) -> float:
        return recall_score(self.data.val.y, self.predictions)

    @cached_property
    def precision(self) -> float:
        return precision_score(self.data.val.y, self.predictions)

    @cached_property
    def f1(self) -> float:
        return f1_score(self.data.val.y, self.predictions)

    @cached_property
    def accuracy(self) -> float:
        return accuracy_score(self.data.val.y, self.predictions)

    @cached_property
    def specificity(self) -> float:
        tn, fp, _, _ = confusion_matrix(self.data.val.y, self.predictions).ravel()
        return tn / (tn + fp)

    def stats(self) -> None:
        """Display all evaluation metrics for this model as a GT table."""
        display(
            GT(pd.DataFrame([{
                "model":       self.label,
                "auc":         self.auc,
                "precision":   self.precision,
                "recall":      self.recall,
                "f1":          self.f1,
                "accuracy":    self.accuracy,
                "specificity": self.specificity,
                "threshold":   self.threshold,
            }]))
            .tab_header(title=self.label, subtitle="Validation set metrics")
            .fmt_number(columns=["auc", "precision", "recall", "f1", "accuracy", "specificity", "threshold"], decimals=3)
        )

In [ ]:
class GraphRenderer:
    """Rendering utilities for model evaluation output."""

    COLORS: ClassVar[list[str]] = [
        "black", "steelblue", "darkorange", "seagreen",
        "crimson", "mediumpurple", "saddlebrown", "deeppink", "teal",
    ]

    @staticmethod
    def filter_by_implemented(model: Model) -> bool:
        """Return True if model.auc is available (i.e. _build_search is implemented).

        Args:
            model: Model instance to check.

        Returns:
            True if accessing model.auc raises no NotImplementedError.
        """
        try:
            _ = model.auc
            return True
        except NotImplementedError:
            return False

    @staticmethod
    def plot_roc_curves(models: list[Model]) -> None:
        """Plot ROC curves for a list of models, legend sorted by AUC descending.

        Args:
            models: Model instances to evaluate. Models whose _build_search raises
                NotImplementedError are shown as dotted legend entries without a curve.
        """
        implemented   = list(filter(GraphRenderer.filter_by_implemented, models))
        unimplemented = [m for m in models if m not in implemented]
        ordered = sorted(implemented, key=lambda m: m.auc, reverse=True) + unimplemented

        fig, ax = plt.subplots(figsize=(8, 7))
        for model, color in zip(ordered, GraphRenderer.COLORS):
            if model in implemented:
                ax.plot(model.fpr, model.tpr, lw=2, color=color,
                        label=f"{model.label}   AUC = {model.auc:.3f}")
            else:
                ax.plot([], [], lw=2, color=color, ls=":",
                        label=f"{model.label}   (not yet implemented)")
        ax.plot([0, 1], [0, 1], color="grey", lw=1, ls="--")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title("ROC Curves — All Models")
        ax.legend(loc="lower right", fontsize=8)
        plt.tight_layout()
        plt.show()

## Naive Benchmark

The naive model ignores all features and predicts a single constant probability for every sample — the proportion of serious delinquencies in the training set. This is the best prediction available without using any features, making it the appropriate floor for every subsequent model. Under ROC AUC the benchmark scores exactly 0.5 by construction, since a constant vector imposes no ranking on the validation set. Any model we build must clear this threshold, and we will verify each AUC improvement is statistically significant rather than attributable to sampling variation.

In [ ]:
class NaiveBaseline(Model):
    """Constant predictor equal to the training base rate."""

    label: ClassVar[str] = "Naive baseline"
    data:  ClassVar[Data] = Data.CLEAN

    def _build_search(self) -> None:
        return None

    @cached_property
    def scores(self) -> np.ndarray:
        return np.full(len(self.data.val.y), float(np.mean(self.data.train.y)))

## Logistic Regression

**Authored by:** Yuxuan Huang

Logistic regression is an interpretable linear classifier fit on the scaled feature branch. The implementation below uses `RandomizedSearchCV` to tune the inverse regularisation strength `C` and solver choice. This keeps the model simple enough to interpret while still allowing the amount of shrinkage to be selected from the data.

### Theory

Logistic regression models the probability of serious delinquency with the logistic link:

$$P(y=1 \mid x)=\sigma(w^\top x+b)=\frac{1}{1+e^{-(w^\top x+b)}}$$

Equivalently, the log-odds are linear in the predictors:

$$\log\left(\frac{P(y=1 \mid x)}{1-P(y=1 \mid x)}\right)=w^\top x+b$$

This makes the model interpretable and relatively stable on small tuning samples. Regularisation controls coefficient size during fitting. L2 regularisation penalises large weights smoothly, while L1 regularisation can shrink some coefficients exactly to zero:

$$\min_{w,b}\; -\ell(w,b)+\lambda\lVert w\rVert_2^2 \quad\text{or}\quad \min_{w,b}\; -\ell(w,b)+\lambda\lVert w\rVert_1$$

Because the target class is highly imbalanced, the estimator uses `class_weight="balanced"` so misclassifying the minority default class carries more weight during fitting.

### Hypothesis

Logistic regression should outperform the naive baseline because it uses borrower-level predictors rather than assigning every borrower the same default probability. Since Branch B reduces skewness and standardises feature scales, the model should provide a strong linear benchmark. If its AUC is close to the more flexible models, that would suggest that much of the useful credit-risk signal is captured by approximately additive effects in the transformed features.

In [ ]:
class LogisticRegressionModel(Model):
    """Logistic Regression on Branch B — standardised features.

    Suggested estimator : LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)
    Suggested search    : C (log-uniform), l1_ratio, solver
    """

    label: ClassVar[str] = "Logistic Regression"
    data:  ClassVar[Data] = Data.SCALED

    def _build_search(self) -> RandomizedSearchCV:
        return RandomizedSearchCV(
            LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED),
            param_distributions=[
                {
                    "C": loguniform(1e-5, 1e3),
                    "l1_ratio": [0.0],
                    "solver": ["lbfgs", "liblinear"],
                },
                {
                    "C": loguniform(1e-5, 1e3),
                    "l1_ratio": [1.0],
                    "solver": ["liblinear"],
                },
            ],
            n_iter=TUNE_ITER,
            scoring="roc_auc",
            cv=TUNE_FOLDS,
            random_state=SEED,
            n_jobs=-1,
        )

### Results

The ROC curves below compare logistic regression with the naive benchmark. ROC AUC is the main metric because the positive class is rare: it measures whether the model ranks actual defaults above non-defaults, rather than rewarding a classifier for predicting the majority class. In this run, the tuned logistic model achieves an AUC of about 0.816, far above the 0.500 AUC of the constant-probability benchmark. The selected specification uses `lbfgs` with a moderate regularisation strength (`C` approximately 0.037), indicating that the model benefits from meaningful coefficient shrinkage rather than an almost unregularised fit.

In [ ]:
naive_baseline = NaiveBaseline()
lr_model       = LogisticRegressionModel()

In [ ]:
#| echo: false
GraphRenderer.plot_roc_curves([naive_baseline, lr_model])

### Conclusion

The logistic regression result shows that scaled linear effects contain substantial predictive information for serious delinquency. Its AUC is among the strongest model results in the report despite the model's simple functional form, which makes it a useful benchmark for the later nonlinear methods. This performance suggests that the preprocessing pipeline has made the main risk signals accessible to a linear classifier, and that added model complexity must deliver only a small incremental gain to justify the loss of interpretability.

## Gradient Boosting Model

**Authored By:** Ty Kouri

To implement an ensemble method we have chosen to utilize a gradient boosting model. Boosting models operate by combining several weak learners into a strong learner. This is generally done by training predictors sequentially, with subsequent predictors attempting to correct mistakes made by its predecessor. The gradient boosting method implements this strategy by adding new predictors sequentially that attempt to fit to the residual errors made by the previous predictor. The reason we chose to implement a gradient boosting method over AdaBoost is due to the make-up of our data. AdaBoost's method of reweighting misclassified samples can lead to over correction for repeated cases of misclassification, which is likely to happen with heavily imbalanced data like the credit data we are working with where only around 6% of the data is classified as a default. Gradient boostings more gradual approach will be better suited to handle this imbalance.

### Theory

Gradient Boosting works by iteratively adding models to fit residual errors using the following method:

The first predictor fits the target directly, leaving a residual:

$$\hat{y} = h_1(x) + \varepsilon, \qquad \varepsilon = y - \hat{y}$$

The next predictor fits that residual:

$$\varepsilon = h_2(x) + \varepsilon'$$

Substituting back, the ensemble after two rounds is:

$$\hat{y}' = h_1(x) + h_2(x) + \varepsilon'$$

Repeating this process, with each new predictor $h_i$ fitting the prior residuals, yields the final sum:

$$\hat{y} = \sum_{i=1}^{k} h_i(x)$$

### Hypothesis

Due to the naive baseline being confined to assigning a constant probability, the gradient boosting method is expected to significantly outperform the benchmark when evaluating their AUC scores using the Delong test. As for comparing to our other more robust models, the gradient boosting method is expected to be one of our higher performing models. The residual fitting strategy of the model should make it more adaptable to the class imbalanced nature of our dataset which might cause other models to struggle.

In [ ]:
GB_PARAMS: dict = {
    "n_estimators":  randint(100, 500),
    "max_depth":     randint(2, 6),
    "learning_rate": loguniform(1e-2, 3e-1),
    "subsample":     uniform(0.6, 0.4),
}

In [ ]:
class GradientBoostingModel(Model):
    """Gradient Boosting on Branch A — raw, unscaled features.

    Suggested estimator : GradientBoostingClassifier(random_state=SEED)
    Suggested search    : n_estimators, max_depth, learning_rate, subsample
    """

    label: ClassVar[str] = "Gradient Boosting"
    data:  ClassVar[Data] = Data.CLEAN

    def _build_search(self) -> RandomizedSearchCV:
        return RandomizedSearchCV(
            GradientBoostingClassifier(random_state=SEED),
            param_distributions=GB_PARAMS,
            n_iter=TUNE_ITER,
            scoring="roc_auc",
            cv=TUNE_FOLDS,
            random_state=SEED,
            n_jobs=-1,
        )

In [ ]:
gb_model = GradientBoostingModel()

In [ ]:
GraphRenderer.plot_roc_curves([naive_baseline, gb_model])

In [ ]:
gb_model.stats()

### Conclusion

As expected, the gradient boosting method vastly outperforms the naive baseline. Additionally, this method is our top performing model based on raw AUC with other top models falling into the similar ~0.82 AUC range. Given that the optimal model selected a lower learning rate around 0.04 and a larger number of estimators at 370, the model seems to be able to take advantage of gradually fitting new predictors to the residuals of prior ones to accurately rank borrowers by default risk despite the minority of defaulted samples in the training set. 

## Support Vector Machine

**Authored by:** Stephen Wallen

SVMs work by finding a decision boundary that maximizes the gap between classes. Because the boundary is defined by distances between points, features with large numeric ranges can drown out features with smaller ones — making scaling a key preprocessing step.

### Theory

#### Hard Margin vs. Soft Margin

An SVM works by drawing a decision boundary (a hyperplane) between two classes and trying to make the gap on either side of that boundary — the margin — as wide as possible.

The hard-margin SVM requires every training point to sit on the correct side of the margin with no exceptions:

$$J(\mathbf{w},b) = \frac{1}{2}||\mathbf{w}||^{2} \quad\text{subject to}\quad y_n(\mathbf{w}\cdot\mathbf{x}_n + b)\geq 1 \;\;\forall\, n$$

In practice this rarely works — real data has noise and overlapping classes, so a perfect separation usually doesn't exist or badly overfits.

Soft-margin SVM adds a penalty for every point that lands on the incorrect side of the margin. This penalty is known as the hinge loss function. In short, the penalty is zero when a point is correctly classified, and grows linearly the further it strays:

$$J(\mathbf{w},b) = \frac{1}{2}||\mathbf{w}||^{2} + C \sum_{n=1}^N \max\{0,\, 1 - y_n(\mathbf{w}\cdot\mathbf{x}_n + b)\}$$

$C$ controls how much those violations are penalised. A large $C$ forces the model to avoid mistakes at the cost of a narrower, more fragile boundary. A small $C$ allows more violations in exchange for a wider, smoother one. Setting $C$ extremely large (we use $10^6$) drives the hinge terms toward zero, recovering the hard-margin solution — which is why both can be approximated with a single `SVC`.

#### Kernel Technique

Linear boundaries don't work well when classes interact in non-linear ways — which is the case with our imbalanced credit data. As a result, we use the "kernel trick" - it replaces the dot product between two points with a function that measures their similarity in a higher-dimensional space, letting the SVM find a non-linear boundary without ever computing that space directly.

The Gaussian RBF kernel is used here:

$$K(\mathbf{a}, \mathbf{b}) = \exp(-\gamma ||\mathbf{a} - \mathbf{b}||^2)$$

It scores two points purely by distance. $\gamma$ controls how quickly that score decays: high $\gamma$ means only very nearby points influence the boundary; low $\gamma$ gives each point broader reach.

#### Sklearn Implementations

Three estimators in scikit-learn cover the SVM family:

| Estimator | Speed | Kernels | Notes |
|---|---|---|---|
| `SVC` | Slower on large datasets | RBF, linear, polynomial, and more | Most flexible; supports probability output |
| `LinearSVC` | Fast | Linear only | Better when a linear boundary is sufficient and the dataset is large |
| `SGDClassifier(loss="hinge")` | Very fast, works in batches | Linear only | Approximation of a linear SVM; useful when data doesn't fit in memory |

`SVC` is used here because the dataset is small enough after subsampling that its higher training cost is acceptable, and we need the RBF kernel to test whether a non-linear boundary helps.

#### Parameter Choices

Only ~6.7% of borrowers defaulted, so without reweighting the model can look accurate by predicting "no default" for almost everyone. class_weight="balanced" corrects this by making incorrect classifications more costly in terms of the penalty function. Because this parameter only reads the target vector y rather than the features, it works correctly regardless of whether the input has been power-transformed or scaled.

$C$ and $\gamma$ are searched on a log-uniform scale because their optimal values can span several orders of magnitude — the difference between 0.1 and 100 is meaningful, while the difference between 100 and 101 is not.

### Hypothesis

We predict the following ranking from worst to best:

1. Hard margin, raw — untreated features let magnitude dominate the kernel, and no slack means the boundary is brittle against noise.
2. Hard margin, transformed — scaling removes the magnitude bias, but the rigid constraint still leaves no room to handle overlap.
3. Soft margin, linear kernel — tuning C adds the flexibility needed to generalise, though a straight-line boundary likely misses non-linear relationships in the data.
4. Soft margin, RBF kernel — expected best, combining scaling, soft margin tolerance, and a kernel that can capture non-linear interactions between borrower features.

In [ ]:
SVM_HARD_C: float = 1e6   # approximates hard margin (C → ∞)

SVM_RBF_PARAMS: dict = {
    "C":     loguniform(1e-2, 1e3),
    "gamma": loguniform(1e-4, 1e1),
}

SVM_LINEAR_PARAMS: dict = {
    "C": loguniform(1e-2, 1e3),
}

In [ ]:
class SVMHard(Model):
    """Hard-margin SVM approximation (C=SVM_HARD_C, RBF kernel).

    Overrides estimator directly — no RandomizedSearchCV.
    Subclasses override label and data to select the preprocessing branch.
    """

    def _build_search(self) -> None:
        return None

    @cached_property
    def estimator(self) -> SVC:
        idx = np.random.RandomState(SEED).choice(
            len(self.data.train.X), size=min(TUNE_SAMPLE, len(self.data.train.X)), replace=False,
        )
        svc = SVC(
            C=SVM_HARD_C,
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=SEED,
            cache_size=2000,
        )
        svc.fit(self.data.train.X.iloc[idx], self.data.train.y.iloc[idx])
        return svc

In [ ]:
class SVMSoft(Model):
    """Soft-margin SVM with tuned C via RandomizedSearchCV.

    Subclasses override label, data, and _kernel to select the preprocessing
    branch and fix the kernel. gamma is included in the search only for RBF.
    """

    _kernel: ClassVar[str] = "rbf"

    def _build_search(self) -> RandomizedSearchCV:
        params = SVM_RBF_PARAMS if self._kernel == "rbf" else SVM_LINEAR_PARAMS
        return RandomizedSearchCV(
            SVC(kernel=self._kernel, probability=True, class_weight="balanced", random_state=SEED, cache_size=2000),
            param_distributions=params,
            n_iter=TUNE_ITER,
            scoring="roc_auc",
            cv=TUNE_FOLDS,
            random_state=SEED,
            n_jobs=-1,
        )

In [ ]:
class SVMHardRaw(SVMHard):
    label: ClassVar[str] = "SVM — Hard margin, Raw"
    data:  ClassVar[Data] = Data.CLEAN


class SVMHardScaled(SVMHard):
    label: ClassVar[str] = "SVM — Hard margin, Scaled"
    data:  ClassVar[Data] = Data.SCALED


class SVMSoftScaled(SVMSoft):
    label: ClassVar[str] = "SVM — Soft margin, Scaled, RBF"
    data:  ClassVar[Data] = Data.SCALED
    _kernel: ClassVar[str] = "rbf"


class SVMSoftScaledLinear(SVMSoft):
    label: ClassVar[str] = "SVM — Soft margin, Scaled, Linear"
    data:  ClassVar[Data] = Data.SCALED
    _kernel: ClassVar[str] = "linear"

In [ ]:
svm_hard_raw    = SVMHardRaw()
svm_hard_scaled = SVMHardScaled()
svm_soft_rbf    = SVMSoftScaled()
svm_soft_linear = SVMSoftScaledLinear()

### Results

In [ ]:
#| echo: false
GraphRenderer.plot_roc_curves([svm_hard_raw, svm_hard_scaled, svm_soft_rbf, svm_soft_linear])

In [ ]:
#| echo: false
svm_hard_raw.stats()
svm_soft_rbf.stats()

### Conclusion

Two of the four predictions were wrong. The soft-margin models did outperform the hard-margin models as expected, and the soft-margin RBF variant improved across every metric — better at catching defaulters and fewer false alarms. But scaling hurt the hard-margin model rather than helping it: the raw-feature variant outperformed the scaled one, likely because the unscaled feature magnitudes happen to produce a more separable space when the boundary has no slack. The bigger surprise was the kernel comparison — the linear soft-margin model beat the RBF one, meaning a straight-line boundary was sufficient once the features were scaled and C was tuned. The non-linear kernel didn't help, which suggests the relationships between these features and default risk are closer to linear than expected in the scaled space.

## Multi-layer Perceptron

**Authored by:** Yuxuan Huang

The multi-layer perceptron (MLP) is a nonlinear classifier fit on the scaled feature branch. The implementation below uses `RandomizedSearchCV` to tune network width and depth through `hidden_layer_sizes`, L2 regularisation through `alpha`, optimisation speed through `learning_rate_init`, and the hidden-layer `activation` function.

### Theory

The MLP learns nonlinear combinations of the standardised features. A one-hidden-layer version can be written as:

$$\hat{y}=\sigma\left(W_2\,g(W_1x+b_1)+b_2\right)$$

Here $g(\cdot)$ is the hidden-layer activation function, such as ReLU or tanh. The hidden units can represent feature interactions that a linear classifier cannot express directly, but the model is also sensitive to regularisation, learning rate, and overfitting. For that reason, the search tunes network size, L2 penalty, learning rate, and activation function, while early stopping holds back part of the training fold and stops fitting when validation performance no longer improves.

### Hypothesis

The MLP should outperform the naive baseline because it uses borrower-level features rather than a constant default probability. It may also outperform logistic regression if credit-risk signal depends on nonlinear interactions among the transformed predictors. However, because the positive class is rare and tuning is performed on a limited training sample, the network may underperform the linear benchmark if the additional flexibility mostly adds estimation noise.

In [ ]:
class MLPModel(Model):
    """Multi-layer Perceptron on Branch B — standardised features.

    Suggested estimator : MLPClassifier(random_state=SEED, max_iter=300)
    Suggested search    : hidden_layer_sizes, alpha, learning_rate_init, activation
    """

    label: ClassVar[str] = "MLP"
    data:  ClassVar[Data] = Data.SCALED

    def _build_search(self) -> RandomizedSearchCV:
        return RandomizedSearchCV(
            MLPClassifier(random_state=SEED, max_iter=300, early_stopping=True),
            param_distributions={
                "hidden_layer_sizes": [(16,), (32,), (64,), (128,), (32, 16), (64, 32), (128, 64)],
                "alpha": loguniform(1e-6, 1e0),
                "learning_rate_init": loguniform(1e-5, 1e-1),
                "activation": ["relu", "tanh", "logistic"],
            },
            n_iter=TUNE_ITER,
            scoring="roc_auc",
            cv=TUNE_FOLDS,
            random_state=SEED,
            n_jobs=-1,
        )

### Results

The ROC curves below compare the MLP with the naive benchmark. Its AUC can then be interpreted alongside the logistic regression result reported earlier. ROC AUC is the main metric because the positive class is rare: it measures whether the model ranks actual defaults above non-defaults, rather than rewarding a classifier for predicting the majority class. In this run, the tuned MLP reaches an AUC of about 0.799. The selected network is compact, using one hidden layer with 32 units, a logistic activation, a very small L2 penalty, and a learning rate of roughly 0.0077.

In [ ]:
mlp_model = MLPModel()

In [ ]:
#| echo: false
GraphRenderer.plot_roc_curves([naive_baseline, mlp_model])

### Conclusion

The MLP clearly improves on the naive benchmark, so the scaled borrower features contain enough information for a nonlinear neural classifier to rank risk above chance. However, its AUC is lower than the logistic regression result, suggesting that the added nonlinear flexibility does not provide a clear advantage under the current tuning budget and class imbalance. In this setting, the simpler logistic model appears to capture the dominant ranking signal more efficiently, while the MLP remains useful as evidence that more complex function classes are not automatically better for this dataset.

### Decision Trees

**Authored by:** Aislinn O'Connell

Decision trees are a machine learning algorithm that utilizes a tree to make predictions. Inputted data traverses through decision nodes that evaluate if the data is greater than or less than the threshold value. This process is repeated recursively until a final prediction is reached.

### Theory

The CART (Classification and Regression Tree) algorithm trains the decision trees. Initially, the dataset is split into two subsets with a single feature, k, and a threshold $t_k$. The algorithm searches for a pair ($k, t_k$), that produces the purest subset, which is evaluated by the Gini Impurity Equation. The Gini impurity of the node informs the user whether all training instances it applies to belong in the same class. A "pure" Gini score is 0.

Gini Impurity Equation:
$$ G_i = 1 - \sum_{k=1}^{n} p_{i,k}^2 $$

This process of splitting the dataset into two and finding the purest subset, is repeated on the split subsets. This recursive process will continue until the maximum depth is achieved, or it cannot be split in a way that will reduce the Gini Impurity score.

CART Cost Function for Classification:
$$ J(k, t_k) = \frac{m_{left}}{m}G_{left} + \frac{m_{right}}{m}G_{right} $$

where

\begin{cases}
G_{left/right} \text{ measures the impurity of the left/right subset}\\
m_{left/right} \text{ number of instances in left/right subset} \\
m = m_{left} + m_{right}
\end{cases}

### Hypothesis

Decision Trees should outperform the naive baseline due to their ability to recognize nonlinear relationships, and interactions between variables.

### Results

Decision trees naturally have high variance, so changing hyperparameters can create substantially different models. Using RandomizedSearchCV(), random parameter combinations were tested and evaluated using cross-validation with TUNE_FOLDS folds over TUNE_ITER iterations, and the model with the highest average ROC AUC score was selected. The highest average ROC AUC score is **0.7926** for this model, which is considered a strong score for an imbalanced credit dataset, and the parameters chosen were:

Optimized Decision Tree Model:
DecisionTreeClassifier(class_weight='balanced', criterion='log_loss',max_depth=5, min_sample)

In [ ]:
class DecisionTreeModel(Model):
    """Decision Tree on Branch A — raw, unscaled features.

    Suggested estimator : DecisionTreeClassifier(class_weight="balanced", random_state=SEED)
    Suggested search    : max_depth, min_samples_split, min_samples_leaf, criterion
    """

    label: ClassVar[str] = "Decision Tree"
    data:  ClassVar[Data] = Data.CLEAN

    def _build_search(self) -> RandomizedSearchCV:
        return RandomizedSearchCV(
            DecisionTreeClassifier(class_weight="balanced", random_state=SEED),
            param_distributions={
                "max_depth": randint(2, 21),
                "min_samples_split": randint(2, 31),
                "min_samples_leaf": randint(1, 16),
                "criterion": ["gini", "entropy", "log_loss"],
            },
            n_iter=TUNE_ITER,
            scoring="roc_auc",
            cv=TUNE_FOLDS,
            random_state=SEED,
            n_jobs=-1,
        )

### Random Forest

**Authored by:** Aislinn O'Connell

Random Forests are a machine learning algorithm that relies on an ensemble of decision trees that searches for the best feature among a random subset of features.

### Theory

The ensemble of decision trees within a Random Forest are typically trained with the bagging (Bootstrap Aggregating) algorithm. Bagging repeatedly generates bootstrap samples by sampling observations from the training set at random with replacement, the same observation(row) can be used multiple times within a single tree's training sample. At each split, a random subset of features are chosen, to ensure that the decision trees do not form strong correlations due to becoming over accustom to the data. This approach for the Random Forest differs from the decision tree model that compares every features. Each individual decision tree generates a prediction independently, and the prediction from the Random Forest is found through majority voting across all trees 

Random Forests improves upon a single Decision Tree by reducing variance while preserving low bias. By averaging predictions across many individual decision trees, even if these trees are overfitting the training data, will result in a more stable model.

### Hypothesis

The Random Forest algorithm will outperform the naive baseline, as well as the singular Decision Tree model. This is because Random Forest combines the predictions from multiple random Decision Trees, and adds randomness into the process through bootstrap sampling and random feature selection. This will reduce overfitting and improve performance overall. It is predicted that the Random Forest will have a higher ROC AUC score when predicting serious delinquency within two years than the Decision Tree and Naive Baseline.

### Results

Random Forest was tuned using RandomizedSearchCV(), where random parameter combinations were evaluated using cross-validation with TUNE_FOLDS folds over TUNE_ITER iterations. The Random Forest model outperformed the naive baseline, but barely outperformed the Decision Tree model. The ROC AUC score for the Random Forest is **0.7985**, compared to the score of **0.7926** for the Decision Tree.

The Random Forest and Decision Tree models, despite having similar ROC AUC scores, had different parameters. The Decision Tree minimized overfitting by staying smaller and avoiding additional complexity with a max depth of 5, a larger minimum leaf size of 11, and a minimum split requirement of 25.

Comparatively, the Random Forest had a maximum depth of 20 and a minimum split threshold of 16, which allowed the ensemble of individual decision trees to become more complex. This reduces overfitting by reducing variance through averaging predictions across multiple flexible decision trees.

Despite differing methodologies to reduce overfitting, the Random Forest and Decision Tree models achieved similar results.

Best Parameters for the Random Forest: RandomForestClassifier(class_weight='balanced', max_depth=20, min_samples_split=16, n_estimators=206, n_jobs=-1, random_state=42)




In [ ]:
class RandomForestModel(Model):
    """Random Forest on Branch A — raw, unscaled features.

    Suggested estimator : RandomForestClassifier(class_weight="balanced", random_state=SEED)
    Suggested search    : n_estimators, max_depth, min_samples_split, max_features
    """

    label: ClassVar[str] = "Random Forest"
    data:  ClassVar[Data] = Data.CLEAN

    def _build_search(self) -> RandomizedSearchCV:
        return RandomizedSearchCV(
            RandomForestClassifier(class_weight="balanced", random_state=SEED, n_jobs=-1),
            param_distributions={
                "n_estimators": randint(100, 501),
                "max_depth": randint(2, 31),
                "min_samples_split": randint(2, 31),
                "max_features": ["sqrt", "log2", None],
            },
            n_iter=TUNE_ITER,
            scoring="roc_auc",
            cv=TUNE_FOLDS,
            random_state=SEED,
            n_jobs=-1,
        )

In [ ]:
dt_model = DecisionTreeModel()
rf_model = RandomForestModel()

### Final ROC AUC Curves

Below is a consolidated plot containing all the ROC plots for each tested models as well as their associated AUC scores.

In [ ]:
MODELS: list[Model] = [
    naive_baseline,
    svm_hard_raw, svm_hard_scaled, svm_soft_rbf, svm_soft_linear,
    lr_model,
    gb_model,
    dt_model,
    rf_model,
    mlp_model,
]

In [ ]:
#| echo: false
GraphRenderer.plot_roc_curves(MODELS)

Looking at the ROC plot above, we can see that our top performing model when measured under raw AUC is the Gradient Boosting Model. Other comparable top models were the Soft-Margin SVM model, the Logistic Regression model, and the Random Forest model. It is interesting to note that the Logistic Regression is our second best model for raw score, with an AUC score that is almost indistinguishable from the top Gradient Boosting model, considering this is the simplest model we implemented besides the naive baseline. This may suggest that a linear decision boundary for classifying defaulted samples is adequate and that, for the sake of computational efficiency and speed, other more complex models may be unnecessary. In the next section we will explore if the difference in AUC scores between our different models can be considered statistically significant.

## Feature Importance

Below is a table showing the feature importance of our tree-based models, giving an idea of which aspects of the feature set are particularly important for predicting whether an individual will have a serious delinquency in the next 2 years. We focus specifically on the tree-based models because they expose a built-in impurity-based importance measure that is intuitive to interpret, given its tie to the tangible mean decrease in impurity at each split.

In [ ]:
tree_models = [
    next(m for m in MODELS if isinstance(m, DecisionTreeModel)),
    next(m for m in MODELS if isinstance(m, RandomForestModel)),
    next(m for m in MODELS if isinstance(m, GradientBoostingModel)),
]

importance_frames = [
    pd.DataFrame({
        "feature": model.data.train.X.columns,
        model.label: model.estimator.feature_importances_,
    }).set_index("feature")
    for model in tree_models
]

importance_matrix = pd.concat(importance_frames, axis=1)
importance_matrix = importance_matrix.loc[
    importance_matrix.max(axis=1).sort_values(ascending=False).index
]

(
    GT(importance_matrix.reset_index(), rowname_col="feature")
    .tab_header(
        title="Feature Importance Across Tree-Based Models",
        subtitle="Impurity-based importance",
    )
    .fmt_number(
        columns=[m.label for m in tree_models],
        decimals=4,
    )
    .data_color(
        columns=[m.label for m in tree_models],
        palette=["white", "steelblue"],
        domain=[0, importance_matrix.values.max()],
    )
)

Looking at the table above, we can see that all three tree-based models concur on the top 3 most important features that were the most influential in reducing impurity. The revolving utilization of unsecured lines seems to be the most influential feature in determining whether an individual is likely to default on their credit payments. Logically, this is quite a strong indicator, as this feature is a measure of an individual's current credit card and personal credit balance divided by their total credit limit. It makes sense that this value would be quite influential, as individuals with large amounts of existing outstanding credit relative to their limit are likely to have a lower probability of being able to meet their payments in the future. The other 2 most important features were the number of times 90 days late and the individual's age. It is worth noting that for the random forest, the number of times 90 days late and age had quite similar importance values, while gradient boosting and the decision tree both assigned more importance to the number of times 90 days late. It is also worth noting that our top performer, the gradient boosting model, assigned over 75% of its importance to just the revolving utilization of unsecured lines and number of times 90 days late features. From the perspective of a credit lender, it makes sense that these two factors are major drivers in calculating FICO scores, which are core to evaluating individual credit risk.

## Feature Requirement: Pairwise DeLong Test

### Purpose And Method

ROC AUC differences should be tested rather than interpreted from the ROC plot alone. Since all models are scored on the same validation borrowers, their AUC estimates are correlated. This feature uses a two-sided DeLong test to compare any two models, with the primary report comparing implemented models against `NaiveBaseline()`.

For models $A$ and $B$:

$$H_0: \operatorname{AUC}_A - \operatorname{AUC}_B = 0$$

$$H_1: \operatorname{AUC}_A - \operatorname{AUC}_B \neq 0$$

A positive difference favours model $A$; a p-value below $\alpha = 0.05$ indicates a statistically significant difference.

### Interface

```python
DeLongTest.compare(LogisticRegressionModel(), NaiveBaseline(), alpha=0.05)
DeLongTest.compare(GradientBoostingModel(), LogisticRegressionModel(), alpha=0.05)
DeLongTest.compare_to_naive(MODELS, alpha=0.05)
```

### Output And Checks

The result table reports both AUCs, their signed difference, the DeLong z-statistic, two-sided p-value, significance flag, and a short interpretation. Comparisons must use continuous scores on identical validation observations. Invalid scores or mismatched labels raise an error; unimplemented models are skipped in the baseline report; zero-variance comparisons return an explicit inference-unavailable result.


## Pairwise DeLong Test Implementation

`DeLongTest.compare` implements the required arbitrary two-model comparison and returns a one-row results table. `DeLongTest.compare_to_naive` applies the same test to each implemented model in `MODELS`, using the naive baseline as the reference. The implementation validates paired validation observations and score arrays before calculating correlated ROC AUC variance.

In [ ]:
class DeLongTest:
    """Two-sided DeLong test for two correlated validation ROC AUCs."""

    RESULT_COLUMNS: ClassVar[list[str]] = [
        "model_a", "model_b", "auc_a", "auc_b", "auc_difference",
        "z_statistic", "p_value", "alpha", "significant", "interpretation",
    ]

    @staticmethod
    def _midrank(values: np.ndarray) -> np.ndarray:
        """Calculate one-based midranks, assigning equal ranks to tied scores."""
        order = np.argsort(values)
        sorted_values = values[order]
        ranks = np.empty(len(values), dtype=float)
        start = 0
        while start < len(values):
            end = start + 1
            while end < len(values) and sorted_values[end] == sorted_values[start]:
                end += 1
            ranks[start:end] = 0.5 * (start + end - 1) + 1
            start = end
        midranks = np.empty(len(values), dtype=float)
        midranks[order] = ranks
        return midranks

    @classmethod
    def _fast_delong(
        cls,
        predictions: np.ndarray,
        positive_count: int,
    ) -> tuple[np.ndarray, np.ndarray]:
        """Estimate correlated AUCs and their covariance using DeLong midranks."""
        model_count, sample_count = predictions.shape
        negative_count = sample_count - positive_count
        positive_scores = predictions[:, :positive_count]
        negative_scores = predictions[:, positive_count:]

        positive_ranks = np.empty((model_count, positive_count), dtype=float)
        negative_ranks = np.empty((model_count, negative_count), dtype=float)
        combined_ranks = np.empty((model_count, sample_count), dtype=float)
        for index in range(model_count):
            positive_ranks[index] = cls._midrank(positive_scores[index])
            negative_ranks[index] = cls._midrank(negative_scores[index])
            combined_ranks[index] = cls._midrank(predictions[index])

        aucs = (
            combined_ranks[:, :positive_count].sum(axis=1)
            / (positive_count * negative_count)
            - (positive_count + 1) / (2 * negative_count)
        )
        positive_components = (
            combined_ranks[:, :positive_count] - positive_ranks
        ) / negative_count
        negative_components = 1 - (
            combined_ranks[:, positive_count:] - negative_ranks
        ) / positive_count
        covariance = (
            np.atleast_2d(np.cov(positive_components)) / positive_count
            + np.atleast_2d(np.cov(negative_components)) / negative_count
        )
        return aucs, covariance

    @staticmethod
    def _validation_labels(model: Model) -> tuple[np.ndarray, pd.Index | None]:
        raw_labels = model.data.val.y
        labels = np.asarray(raw_labels)
        if labels.ndim != 1:
            raise ValueError(f"{model.label} validation labels must be one-dimensional.")
        if not np.isfinite(labels.astype(float)).all():
            raise ValueError(f"{model.label} validation labels contain non-finite values.")
        return labels, getattr(raw_labels, "index", None)

    @staticmethod
    def _model_scores(model: Model) -> np.ndarray:
        try:
            scores = np.asarray(model.scores, dtype=float)
        except NotImplementedError as error:
            raise NotImplementedError(
                f"{model.label} is not implemented and cannot be compared."
            ) from error
        if scores.ndim != 1:
            raise ValueError(f"{model.label} scores must be one-dimensional.")
        if not np.isfinite(scores).all():
            raise ValueError(f"{model.label} scores contain non-finite values.")
        return scores

    @classmethod
    def compare(
        cls,
        model_a: Model,
        model_b: Model,
        alpha: float = 0.05,
    ) -> pd.DataFrame:
        """Run a two-sided DeLong test for any two models on paired validation data."""
        if not 0 < alpha < 1:
            raise ValueError("alpha must be between 0 and 1.")

        labels_a, index_a = cls._validation_labels(model_a)
        labels_b, index_b = cls._validation_labels(model_b)
        if index_a is not None and index_b is not None and not index_a.equals(index_b):
            raise ValueError("Models must use validation observations in the same order.")
        if not np.array_equal(labels_a, labels_b):
            raise ValueError("Models must use identical validation target labels.")
        if set(np.unique(labels_a).tolist()) != {0, 1}:
            raise ValueError("DeLong testing requires binary validation labels encoded as 0 and 1.")

        scores_a = cls._model_scores(model_a)
        scores_b = cls._model_scores(model_b)
        if len(scores_a) != len(labels_a) or len(scores_b) != len(labels_a):
            raise ValueError("Each score array must match the validation target length.")

        positive_count = int(np.sum(labels_a == 1))
        negative_count = len(labels_a) - positive_count
        if positive_count < 2 or negative_count < 2:
            raise ValueError("DeLong testing requires at least two observations in each class.")

        order = np.argsort(-labels_a, kind="stable")
        predictions = np.vstack([scores_a, scores_b])[:, order]
        aucs, covariance = cls._fast_delong(predictions, positive_count)
        difference = float(aucs[0] - aucs[1])
        contrast = np.array([1.0, -1.0])
        variance = float(contrast @ covariance @ contrast)
        if variance < -1e-12:
            raise ValueError("DeLong covariance produced an invalid negative variance.")
        variance = max(variance, 0.0)

        if variance <= np.finfo(float).eps:
            z_statistic = np.nan
            p_value = np.nan
            significant = False
            interpretation = (
                "Inference unavailable because the estimated variance of the AUC "
                "difference is zero."
            )
        else:
            z_statistic = difference / np.sqrt(variance)
            p_value = float(2 * norm.sf(abs(z_statistic)))
            significant = bool(p_value < alpha)
            if significant:
                higher_auc_model = model_a.label if difference > 0 else model_b.label
                interpretation = (
                    f"{higher_auc_model} has a statistically significantly higher "
                    f"AUC at alpha = {alpha:.3g}."
                )
            else:
                interpretation = f"No statistically significant AUC difference at alpha = {alpha:.3g}."

        return pd.DataFrame([{
            "model_a": model_a.label,
            "model_b": model_b.label,
            "auc_a": float(aucs[0]),
            "auc_b": float(aucs[1]),
            "auc_difference": difference,
            "z_statistic": z_statistic,
            "p_value": p_value,
            "alpha": alpha,
            "significant": significant,
            "interpretation": interpretation,
        }], columns=cls.RESULT_COLUMNS)

    @classmethod
    def compare_to_naive(
        cls,
        models: list[Model],
        alpha: float = 0.05,
    ) -> pd.DataFrame:
        """Compare every implemented non-baseline model with the naive benchmark."""
        baseline = next(
            (model for model in models if isinstance(model, NaiveBaseline)),
            None,
        )
        if baseline is None:
            baseline = NaiveBaseline()
        comparisons: list[pd.DataFrame] = []
        for model in models:
            if isinstance(model, NaiveBaseline):
                continue
            try:
                comparisons.append(cls.compare(model, baseline, alpha=alpha))
            except NotImplementedError:
                print(f"Skipping {model.label}: model is not yet implemented.")
        if not comparisons:
            return pd.DataFrame(columns=cls.RESULT_COLUMNS)
        return pd.concat(comparisons, ignore_index=True)


    @classmethod
    def compare_all_pairs(
        cls,
        models: list[Model],
        alpha: float = 0.05,
        include_baseline: bool = False,
    ) -> pd.DataFrame:
        """Run pairwise DeLong tests for every unique pair of implemented models."""

        eligible = [
            model for model in models
            if include_baseline or not isinstance(model, NaiveBaseline)
        ]
        comparisons: list[pd.DataFrame] = []
        for m1, m2 in combinations(eligible, 2):
            try:
                comparisons.append(cls.compare(m1, m2, alpha=alpha))
            except NotImplementedError:
                print(f"Skipping pair ({m1.label}, {m2.label}): not implemented.")
        if not comparisons:
            return pd.DataFrame(columns=cls.RESULT_COLUMNS)
        return pd.concat(comparisons, ignore_index=True)

### Results Against The Naive Baseline

The table below runs the pairwise Delong test for each model against the naive benchmark. 

In [ ]:
#| echo: false
delong_against_naive = DeLongTest.compare_to_naive(MODELS)
(
    GT(delong_against_naive)
    .tab_header(
        title="Pairwise DeLong Tests Against The Naive Baseline",
        subtitle="Two-sided tests of correlated validation ROC AUCs",
    )
    .fmt_number(
        columns=["auc_a", "auc_b", "auc_difference", "z_statistic", "p_value", "alpha"],
        decimals=4,
    )
)

Under the DeLong test, two models are said to have a statistically significant difference in AUC when the test produces a p-value of $p \leq 0.05$. As can be seen in the table above, all 8 evaluated models had a statistically significant outperformance of the naive benchmark when evaluated by AUC. This confirms that all models' AUC scores are statistically distinguishable from 0.5 to suggest that they are able to derive information from the features of the dataset in order to make assessments on the probability of a default relative to a constant probability model. 

### Pairwise Delong Tests Between Implemented Models

The table below runs a pairwise Delong Test for each combination of implemented models in a grid layout and reports their test p-value. This allows us to compare if model ROC performance is meaningfully different between more complex models. 

In [ ]:
def pairwise_p_value_grid(results):
    """Symmetric p-value grid from long-format DeLong results."""
    labels = list(dict.fromkeys(results["model_a"].tolist() + results["model_b"].tolist()))
    grid = pd.DataFrame(index=labels, columns=labels, dtype=float)
    for _, row in results.iterrows():
        grid.loc[row["model_a"], row["model_b"]] = row["p_value"]
        grid.loc[row["model_b"], row["model_a"]] = row["p_value"]
    for label in labels:
        grid.loc[label, label] = np.nan
    return grid


delong_all_pairs = DeLongTest.compare_all_pairs(MODELS)
p_value_grid = pairwise_p_value_grid(delong_all_pairs).reset_index(names="model")

value_cols = [c for c in p_value_grid.columns if c != "model"]

(
    GT(p_value_grid, rowname_col="model")
    .tab_header(title="Pairwise DeLong p-values")
    .fmt_scientific(columns=value_cols, decimals=2)
    .sub_missing(missing_text="—")
)


For the sake of simplicity, we will focus primarily on the DeLong test p-values for the Gradient Boosting model, since this was our top-performing model when measured by raw AUC score. As can be seen above, the Gradient Boosting model's AUC score was statistically significantly different from both SVM Hard-Margin models, the Random Forest, Decision Tree, and MLP models. Therefore, we can say that the Gradient Boosting model had a statistically significant outperformance of these models in evaluating the probability that an individual will have a serious credit delinquency in the next 2 years.

The AUC scores of the SVM Soft-Margin and Logistic Regression models, however, were not statistically significantly different from that of the Gradient Boosting model. These were our top 3 models when ranked by raw AUC score, and the DeLong tests seem to suggest that their difference in performance is small enough to be considered within noise. As such, we think that all three of these models would be adequate choices for forecasting the probability that an individual may experience a credit delinquency in the next 2 years when provided with the same feature set we analyzed in this project. Given that the Logistic Regression and Gradient Boosting models have incredibly fast run times relative to their peers, these models may be the best choice for a larger-scale application, since their expediency will be well suited to evaluating large groups of potential borrowers.

## Conclusion

Throughout the course of our analysis, we have explored 8 different candidate models and assessed their performance in accurately providing probabilities for a serious credit delinquency in the next 2 years. The models did so by training on a sample set containing 9 different credit-relevant features after our data preprocessing. Based on our analysis, the two most successful models for achieving this goal were the Logistic Regression and Gradient Boosting models. These two models were our top performers when measured by AUC and had the added benefit of having quick run times that lend themselves well to scaling up to scenarios where large quantities of borrowers need to be evaluated. These models were also found to have a statistically significant difference in performance from most of the other evaluated models when AUC scores were compared using a DeLong test. Therefore, in a credit lending context, these two models serve as our recommendation for evaluating the credit risk of potential borrowers. A possible area for further exploration that we would like to recommend is a consolidated stacking model which utilizes the forecasts of both models to create a potentially more robust default prediction.

## Extra Credit - Threshold Tuning & Corresponding Detection Performance

The model with the best AUC score was the Gradient Boosting Model with an AUC of 0.817432. The optimal threshold for the Gradient Boosting model was found to be 0.03956, which is significantly lower than the default threshold of 0.50. Having the optimized threshold at a much lower value than the default value indicates that classifying borrowers as high risk despite the estimated probability of default still being fairly low improves the Gradient Boosting model's detection performance. The Logistic Regression model, which had the second-highest AUC score (0.815757), had an optimal threshold of 0.508, which is essentially the same as the usual threshold of 0.5. Threshold tuning had a much larger impact on the Gradient Boosting model than it did the Logistic Regression model.

Looking at the confusion matrices for Gradient Boosting, one with the optimized threshold and one without, it is evident that threshold tuning increased the model's recall, but diminished the precision. The model with the default threshold of 0.5 had a recall score of 0.1554 and a precision score of 0.4752, while the model with the optimized threshold had a recall score of 0.7444 and a precision score of 0.1738. The default threshold Gradient Boosting model correctly predicted 307 borrowers who experienced serious delinquency, and the optimized threshold version correctly predicted 1,471 borrowers who experienced delinquency. However, the improved recall of the optimized threshold came at the cost of 6,994 false positive classifications, while the model with the default threshold made only 339 false positive classifications. This perfectly represents the recall-precision tradeoff.

The F1 score and accuracy also show the effects of threshold tuning. By optimizing the threshold for the Gradient Boosting model the F1 score went from 0.2342 to 0.2818, which shows a slightly better balance between precision and recall. The accuracy for the default model was 93.29% while the optimized threshold model had an accuracy of 74.96%. At first glance, it might appear as though the model without the optimal threshold is the better model. However, the model can achieve high accuracy by predicting the majority class (non-delinquent borrowers) while still missing many actual defaults. Therefore, the increase in recall, and F1 score provided by the optimized threshold model may be more valuable for credit risk prediction than maximizing overall accuracy.

In [ ]:
best_model = max(MODELS[1:], key=lambda m: m.auc)

print("Best model:", best_model.label)
print("AUC:", best_model.auc)
print("Optimal threshold:", best_model.threshold)

best_model.stats()

ConfusionMatrixDisplay.from_predictions(
    best_model.data.val.y,
    best_model.predictions
)

plt.title(f"Confusion Matrix Optimal Threshold — {best_model.label}")
plt.show()


y_pred_default = (best_model.scores >= 0.50).astype(int)

ConfusionMatrixDisplay.from_predictions(
    best_model.data.val.y,
    y_pred_default
)
plt.title(f"Confusion Matrix Default 0.50 Threshold — {best_model.label}")
plt.show()

print("Default Threshold = 0.50")
print("Precision:", precision_score(best_model.data.val.y, y_pred_default))
print("Recall:", recall_score(best_model.data.val.y, y_pred_default))
print("F1:", f1_score(best_model.data.val.y, y_pred_default))
print("Accuracy:", accuracy_score(best_model.data.val.y, y_pred_default))

threshold_results = pd.DataFrame([
    {
        "model": m.label,
        "auc": m.auc,
        "threshold": m.threshold,
        "precision": m.precision,
        "recall": m.recall,
        "f1": m.f1,
        "accuracy": m.accuracy,
        "specificity": m.specificity,
    }
    for m in MODELS
])

(
    GT(threshold_results.sort_values("auc", ascending=False))
    .fmt_number(
        columns=["auc", "threshold", "precision", "recall", "f1", "accuracy", "specificity"],
        decimals=4,
    )
)

## Citations

- Kaggle. (2011). *Give Me Some Credit* [Data set]. Kaggle. https://www.kaggle.com/competitions/GiveMeSomeCredit
- DeLong, E. R., DeLong, D. M., & Clarke-Pearson, D. L. (1988). Comparing the areas under two or more correlated receiver operating characteristic curves: A nonparametric approach. *Biometrics, 44*(3), 837-845. https://doi.org/10.2307/2531595 PMID: 3203132.
- Wikipedia contributors. (n.d.). D'Agostino's K-squared test. In *Wikipedia, The Free Encyclopedia*. Retrieved June 4, 2026, from https://en.wikipedia.org/wiki/D%27Agostino%27s_K-squared_test
- Wikipedia contributors. (n.d.). Pearson correlation coefficient. In *Wikipedia, The Free Encyclopedia*. Retrieved June 4, 2026, from https://en.wikipedia.org/wiki/Pearson_correlation_coefficient

### AI Statement

During the preparation of this final report, the authors utilized AI to assist with the structural organization, grammatical flow, and readability of the written prose. In strict accordance with the course's Coding Integrity policy, the team independently authored all core logic, data analysis, and machine learning pipelines. AI tools were utilized exclusively in a permitted support capacity to interpret error messages, debug syntax issues, and conceptually validate the modeling approach. No AI-generated code solutions were copied or pasted into the final repository. The authors have thoroughly reviewed all outputs and assume full responsibility for the accuracy and integrity of this submission.
